## Notebook setup

In [1]:
import sys
sys.path.insert(0, "../src")

## Analysis

In [3]:
import datetime
import re

In [4]:
import pandas as pd

In [5]:
from core.game import Game
from core.entity import Team

In [ ]:
# @dataclass
# class Game:
#     number: int
#     team1: list[str]
#     team2: list[str]
#     start_time: datetime.datetime
#     end_time: datetime.datetime | None = None
#     team1_score: int | None = None
#     team2_score: int | None = None
#     outcome: str = "in_progress"  # "finished", "abandoned", "in_progress"

In [ ]:
def parse_audit(path: str) -> list[Game]:
    games: dict[int, Game] = {}

    with open(path) as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            parts = line.split(maxsplit=1)
            if len(parts) < 2:
                continue
            ts_str, action = parts
            ts = datetime.datetime.strptime(ts_str, "%Y-%m-%dT%H:%M:%SZ")

            if action.startswith("Start Game "):
                rest = action[len("Start Game "):]
                num_str, teams = rest.split(" ", 1)
                team1_str, team2_str = teams.split(" v ", 1)
                games[int(num_str)] = Game(
                    number=int(num_str),
                    team1=Team(players=[p.strip() for p in team1_str.split(" and ")]),
                    team2=Team(players=[p.strip() for p in team2_str.split(" and ")]),
                    start_time=ts,
                )

            elif action.startswith("Finish Game "):
                rest = action[len("Finish Game "):]
                num = int(rest.split(" ", 1)[0])
                if num in games:
                    m = re.search(r"\((\d+) - (\d+)\)$", rest)
                    games[num].end_time = ts
                    games[num].outcome = "finished"
                    if m:
                        games[num].team1_score = int(m.group(1))
                        games[num].team2_score = int(m.group(2))

            elif action.startswith("Abandon Game "):
                rest = action[len("Abandon Game "):]
                num = int(rest.split(" ", 1)[0])
                if num in games:
                    games[num].end_time = ts
                    games[num].outcome = "abandoned"

    return sorted(games.values(), key=lambda g: g.number)

In [4]:
games = parse_audit("../data/superbadder/20260421/audit.txt")

In [ ]:
def view_audit_data(games: list[Game]) -> pd.DataFrame:
    rows = []
    for g in games:
        duration_min = (
            round((g.end_time - g.start_time).total_seconds() / 60, 1)
            if g.end_time else None
        )
        rows.append({
            "game": g.number,
            "team1": " & ".join(g.team1),
            "team2": " & ".join(g.team2),
            "start_time": g.start_time.strftime("%H:%M:%S"),
            "end_time": g.end_time.strftime("%H:%M:%S") if g.end_time else None,
            "duration_min": duration_min,
            "team1_score": g.team1_score,
            "team2_score": g.team2_score,
            "outcome": g.outcome,
        })

    df = pd.DataFrame(rows)
    return df

,game,team1,team2,start_time,end_time,duration_min,team1_score,team2_score,outcome
0,1,Akhil Shrikanth & Andy,Thenes & Ashutosh,09:07:53,09:16:00,8.1,21.0,14.0,finished
1,2,Barrie & Ulf,Sumit & Mag,09:10:07,09:19:03,8.9,21.0,11.0,finished
2,3,Harry & Ravi,Suandi & Pritto,09:14:14,09:24:42,10.5,22.0,24.0,finished
3,5,Ulf & Mag,Yaw & Lesley,09:21:22,09:30:24,9.0,21.0,18.0,finished
4,6,Thenes & Akhil Shrikanth,Barrie & Sumit,09:21:37,09:33:09,11.5,18.0,21.0,finished
5,7,Andy & Ashutosh,Harry & Pritto,09:25:11,09:34:10,9.0,13.0,21.0,finished
6,8,Suandi & Ravi,Kelvin & Yaw,09:30:42,09:42:21,11.7,12.0,21.0,finished
7,9,Sumit & Ulf,Barrie & Lesley,09:34:18,09:43:55,9.6,21.0,17.0,finished
8,10,Trung & Akhil Shrikanth,Harry & Andy,09:36:50,09:41:54,5.1,6.0,21.0,finished
9,11,Thenes & Pritto,Trung & Ashutosh,09:42:09,09:50:53,8.7,21.0,12.0,finished


In [7]:
def player_id(name: str) -> str:
    return name.lower().replace(" ", "_")

# Upsert all players seen across all games
all_players = {p for g in games for p in g.team1 + g.team2}
cursor.executemany(
    "INSERT OR IGNORE INTO players (player_id, player_name) VALUES (?, ?)",
    [(player_id(p), p) for p in sorted(all_players)],
)

# Insert games
cursor.executemany(
    """
    INSERT INTO games
        (team1_player1, team1_player2, team1_score,
         team2_player1, team2_player2, team2_score,
         start_time, end_time, status)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """,
    [
        (
            player_id(g.team1[0]), player_id(g.team1[1]), g.team1_score,
            player_id(g.team2[0]), player_id(g.team2[1]), g.team2_score,
            g.start_time.isoformat(), g.end_time.isoformat() if g.end_time else None,
            g.outcome,
        )
        for g in games
    ],
)

conn.commit()
print(f"Inserted {len(all_players)} players, {len(games)} games")

Inserted 16 players, 31 games
